# Phase 5: Self-Healing RAG with LangGraph

This notebook runs only free, open-source models inside the Google Colab runtime. It uses no paid APIs or hosted inference endpoints. The Phase 5 thresholds are initial heuristics, not statistically calibrated decisions.

## Environment

Replace `REPOSITORY_URL` with your repository URL before running this notebook in Colab.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORY_URL = 'https://github.com/YOUR_GITHUB_USERNAME/adaptive-rag.git'
PROJECT_DIR = Path('/content/adaptive-rag')

if not (PROJECT_DIR / 'src').exists():
    if 'YOUR_GITHUB_USERNAME' in REPOSITORY_URL:
        raise RuntimeError('Set REPOSITORY_URL to your GitHub repository URL, then run this cell again.')
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print(f'Working directory: {Path.cwd()}')

In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'unavailable')
if not torch.cuda.is_available():
    print('Enable a GPU under Runtime > Change runtime type for practical local Qwen inference.')

## Initialize local retrieval and generation components

The embedding model, cross-encoder, and Qwen model are downloaded from Hugging Face and executed directly in this Colab runtime.

In [ ]:
from src.rag import (
    BM25Retriever,
    CrossEncoderReranker,
    FAISSRetriever,
    HybridRetriever,
    LocalQwenGenerator,
    RAGConfig,
    RetrievalFailureDetector,
    SelfHealingRAGWorkflow,
    SelfHealingWorkflowConfig,
    load_documents,
)

rag_config = RAGConfig(
    embedding_model_name='sentence-transformers/all-MiniLM-L6-v2',
    generation_model_name='Qwen/Qwen2.5-1.5B-Instruct',
    max_new_tokens=220,
)
documents = load_documents(Path('data/phase1_corpus.json'))
index_dir = Path('data/phase1_faiss_index')

dense_retriever = FAISSRetriever(
    embedding_model_name=rag_config.embedding_model_name,
    device=rag_config.device,
    batch_size=rag_config.embedding_batch_size,
)
if (index_dir / 'documents.faiss').exists():
    dense_retriever.load(index_dir)
    print(f'Loaded FAISS index with {len(dense_retriever.documents)} documents.')
else:
    dense_retriever.build(documents)
    dense_retriever.save(index_dir)
    print(f'Built FAISS index with {len(documents)} documents.')

bm25_retriever = BM25Retriever(documents)
hybrid_retriever = HybridRetriever(dense_retriever, bm25_retriever)
reranker = CrossEncoderReranker(device=rag_config.device)
generator = LocalQwenGenerator(rag_config)
print('Reranker device:', reranker.device)
print('Qwen device:', generator.device)

## Configure heuristics and compile the graph

Workflow: `retrieve → rerank → diagnostics → classify`, followed by a conditional route to generation, expanded retrieval, local query rewriting, or abstention. Both recovery routes return to retrieval.

In [ ]:
from dataclasses import asdict

failure_detector = RetrievalFailureDetector()
workflow_config = SelfHealingWorkflowConfig(
    initial_retrieval_depth=5,
    expanded_retrieval_depth=10,
    reranked_top_k=5,
    generation_top_k=3,
    max_retries=1,
)
workflow = SelfHealingRAGWorkflow(
    dense_retriever=dense_retriever,
    bm25_retriever=bm25_retriever,
    hybrid_retriever=hybrid_retriever,
    reranker=reranker,
    generator=generator,
    failure_detector=failure_detector,
    config=workflow_config,
)

print('Initial, uncalibrated failure-detector thresholds:')
for name, value in asdict(failure_detector.config).items():
    print(f'  {name}: {value}')
print('\nWorkflow configuration:')
for name, value in asdict(workflow_config).items():
    print(f'  {name}: {value}')

## Run self-healing examples

The exact classifications may change when the initial thresholds are evaluated in Phase 6. This notebook reports the observed state, reasons, recovery action, path, documents, and final response.

In [ ]:
def display_workflow_result(label: str, query: str):
    state = workflow.run(query)
    low_level_steps = {'RETRIEVE', 'RERANK', 'DIAGNOSTICS'}
    decision_path = [step for step in state['path'] if step not in low_level_steps]

    print('\n' + '=' * 100)
    print('Scenario:', label)
    print('Original query:', state['original_query'])
    print('Rewritten query:', state.get('rewritten_query'))
    print('Decision path:', ' → '.join(decision_path))
    print('Full graph path:', ' → '.join(state['path']))
    print('Failure history:', [item.value for item in state['failure_history']])
    print('Final failure type:', state['failure_type'].value)
    print('Classification reasons:', list(state['classification_reasons']))
    print('Retry count:', state['retry_count'])
    print('Recovery action:', state['recovery_action'].value)
    print('Retrieval depth:', state['retrieval_depth'])

    print('Retrieved documents used for generation:')
    for rank, result in enumerate(state.get('retrieved_documents', []), start=1):
        print(f'  {rank}. {result.title} (id={result.id}, reranker={result.reranker_score:.6f}, RRF={result.rrf_score:.6f})')
    print('\nFinal answer:')
    print(state['final_answer'])
    return state

In [ ]:
query_scenarios = [
    ('Normal supported query', 'Where is the Mona Lisa displayed?'),
    ('Paraphrased query', 'Which orbiting observatory takes its name from an astronomer?'),
    ('Ambiguous query', 'What happened in 1991?'),
    ('Missing-evidence query', 'What is the capital of Canada?'),
]

results = {}
for label, query in query_scenarios:
    results[label] = display_workflow_result(label, query)